# 09 — Final-holdout backtest, ablation, benchmarks

Evaluate the **frozen** shortlist on **2024 only** (`final_idx`).
Vol-matched SPY scale is fit on **selection** strategy returns, then applied to final.

> Combined 2022–2024 analysis is exploratory and is **not** produced here.

In [ ]:
import pandas as pd
from spy_alpha_research.data import (
    load_config, project_root, load_frame, split_three_way,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]
ret_type = bt.get("return_type", "log")

from spy_alpha_research.attribution import load_alpha_candidates, grouped_ablation
from spy_alpha_research.alpha import build_alpha_signals, equal_weight_blend
from spy_alpha_research.reporting import (
    compare_signals, plot_equity_curves, plot_comparison_table_bars, plot_ablation,
)
from spy_alpha_research.backtesting import run_timing_backtest, benchmark_bundle, summarize_benchmarks

feats = load_frame(PROC / "features_raw.parquet")
ret = load_frame(PROC / "spy_returns.parquet").iloc[:, 0]
target = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
train_idx, sel_idx, final_idx = split_three_way(
    feats.index,
    train_end=dates["train_end"],
    selection_start=dates["selection_start"],
    selection_end=dates["selection_end"],
    final_start=dates["final_start"],
    end=dates["end"],
)
assert final_idx.min() >= pd.Timestamp(dates["final_start"])
assert final_idx.max() <= pd.Timestamp(dates["end"])

alphas = load_alpha_candidates(ROOT / exp["paths"]["alpha_candidates"])
signals = build_alpha_signals(
    feats, alphas, target, train_idx, min_periods=exp["scaling"]["min_periods"]
)
signals["eqw_alpha"] = equal_weight_blend(signals)
hold = {k: v.reindex(final_idx) for k, v in signals.items()}

cmp = compare_signals(
    hold, ret.reindex(final_idx), target.reindex(final_idx),
    signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    periods_per_year=bt["periods_per_year"],
    ic_window=exp["statistics"]["rolling_ic_window"], nw_lags=exp["alpha"]["nw_lags"],
    return_type=ret_type,
)

eqw_bt = run_timing_backtest(
    hold["eqw_alpha"], ret.reindex(final_idx),
    signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    return_type=ret_type,
)
# Selection-window eqw for vol-matching scale only (never use final for scale)
sel_eqw = run_timing_backtest(
    signals["eqw_alpha"].reindex(sel_idx), ret.reindex(sel_idx),
    signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    return_type=ret_type,
)
bmarks = benchmark_bundle(
    sel_eqw.net_returns, ret,
    selection_index=sel_idx, eval_index=final_idx,
    periods_per_year=bt["periods_per_year"],
    return_type=ret_type,
)
curves = {
    k: run_timing_backtest(
        v, ret.reindex(final_idx),
        signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
        return_type=ret_type,
    ).net_returns
    for k, v in hold.items()
}
curves.update(bmarks)
plot_equity_curves(curves, FIG / "eqw_vs_singles_equity.png", title="Final 2024 net equity + benchmarks")
plot_equity_curves(
    {"eqw_alpha": eqw_bt.net_returns, **bmarks},
    FIG / "benchmarks_equity.png",
    title="Equal-weight vs benchmarks (final 2024)",
)
plot_comparison_table_bars(cmp, "sharpe", FIG / "all_signals_compare.png")

abl = grouped_ablation(
    feats, alphas, target, ret, train_idx, final_idx,
    min_periods=exp["scaling"]["min_periods"],
    signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    return_type=ret_type,
)
plot_ablation(abl, FIG / "ablation_groups.png")
cmp, summarize_benchmarks(bmarks), abl